In [ ]:
import subprocess
import sys
import random as rd
import webbrowser

def instalar(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

try:
    import pandas as pd
except ImportError:
    instalar('pandas')
    import pandas as pd

In [52]:
#Criar backup
try:
    backup = pd.read_csv('save.csv')
except:
    backup = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
backup.to_csv('save_backup.csv', index=False)

In [ ]:
def escolhe_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    for index, row in data.iterrows(): 
        print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]}')

In [54]:
def ver_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            print(f'0{row["id"]} {row["name"]} - {row["grade"]} - {row["clan"]} - {row["type"]} - {row["rarity"]} - Qtt: {row["qtt"]}')
    except:
        print('Você ainda não tem nenhuma carta na coleção! Abra alguns pacotes para começar a colecionar!')

In [55]:
def rodar_pacote(name):
    data = pd.read_csv(f'packs\{name}.csv')
    pacote = []
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity'])

    commons = data[data['rarity'] == 'C']
    c_tiradas = commons.sample(n=4).to_dict('records')
    pacote.extend(c_tiradas)

    luck = rd.randint(1, 100)

    if luck <= 70:
        raridade = 'R'
    elif luck <= 90:
        raridade = 'RR'
    elif luck <= 99:
        raridade = 'RRR'
    else:
        raridade = 'SP'

    raras = data[data['rarity'] == raridade]
    r_tirada = raras.sample(n=1).to_dict('records')
    pacote.extend(r_tirada)

    pacote = pd.DataFrame(pacote)
    
    return pacote

In [56]:
def atualizar_save(box, pacote):
    #Quero colocar os valores do pacote na box e se houver repetidos, qtt += 1
    
    for index, row in pacote.iterrows():
        card = box[(box['set'] == row['set']) & (box['id'] == row['id'])]
        if not card.empty:
            box.loc[card.index, 'qtt'] += 1
        else:
            new_card = row.to_dict()
            new_card['qtt'] = 1
            #box não tem append, então vou criar um novo dataframe com a nova linha e concatenar com a box
            new_card_df = pd.DataFrame([new_card])
            box = pd.concat([box, new_card_df], ignore_index=True)

    return box

In [ ]:
def rodar_box(name, qtt):
    box = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])
    
    for i in range(qtt):
        pacote = rodar_pacote(name)
        box = atualizar_save(box, pacote)
    
    try:
        save = pd.read_csv('save.csv')
    except:
        save = pd.DataFrame(columns=['set', 'id', 'name', 'grade', 'clan', 'type', 'rarity', 'qtt'])

    box.sort_values(by=['id'], inplace=True)
    box.to_csv('last_box.csv', index=False)

    save = atualizar_save(save, box)
    save.sort_values(by=['set', 'id'], inplace=True)
    save.to_csv('save.csv', index=False)
    
    return box

In [58]:
def gerar_link_wiki(nome_carta):
    # A wiki usa underscores no lugar de espaços
    nome_formatado = nome_carta.replace(' ', '_')
    return f'https://cardfight.fandom.com/wiki/{nome_formatado}'

In [61]:
def web_colecao(nome):
    try:
        save = pd.read_csv(f'{nome}.csv')
        for index, row in save.iterrows():
            link = gerar_link_wiki(row['name'])
            print(f'Abrindo Wiki de: {row["name"]}...')
            webbrowser.open(link) # Isso abre o navegador automaticamente
    except:
        print('Erro ao carregar coleção.')

In [64]:
save = pd.read_csv('save.csv')
save[save['clan'] == 'Kagero']

,set,id,name,grade,clan,type,rarity,qtt
0,BT01,5,"Embodiment of Victory, Aleph",3,Kagero,Normal Unit,RRR,1
1,BT01,13,Vortex Dragon,3,Kagero,Normal Unit,RR,1
19,BT01,48,"Embodiment of Armor, Bahr",1,Kagero,Normal Unit,C,1
20,BT01,49,"Dragon Monk, Gojo",1,Kagero,Normal Unit,C,1
21,BT01,50,"Wyvern Strike, Jarran",1,Kagero,Normal Unit,C,1
22,BT01,51,"Dragon Dancer, Monica",0,Kagero,Draw Trigger,C,1
23,BT01,52,"Lizard Soldier, Ganlu",0,Kagero,Stand Trigger,C,1
